# Your First Custom Tool with Claude Agent SDK

Custom tools are plain Python functions you offer to Claude, which it can invoke as needed — this is how you extend the agent beyond the built-in tools.


In [1]:
from typing import Any

from claude_agent_sdk import (
    tool,  # decorator that turns a Python function into a Claude-usable tool
    create_sdk_mcp_server,  # bundles one or more tools into a "server" Claude can talk to
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    AssistantMessage,  # message type that holds Claude's actual reply
    ToolUseBlock,  # message piece that shows "Claude is calling a tool now"
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)

# Pretend price list — stands in for a real stock API so this demo has no external calls.
MOCK_PRICES = {"AAPL": 193.50, "GOOGL": 178.25, "MSFT": 412.80}


def get_stock_price(ticker: str) -> dict[str, float]:
    """Mock stock price lookup — no real API call."""
    return {"price": MOCK_PRICES.get(ticker.upper(), 100.00)}


# @tool(name, description, input_schema) — this is what turns a normal async
# function into something Claude can call like a built-in tool:
#   "get_stock_price"                          -> the tool's name
#   "Get the current mock stock price..."      -> description Claude reads to
#                                                  decide WHEN to use this tool
#   {"ticker": str}                             -> expected input: one argument
#                                                  called "ticker", of type str
@tool(
    "get_stock_price",
    "Get the current mock stock price for a ticker symbol",
    {"ticker": str},
)
async def get_stock_price_tool(args: dict[str, Any]) -> dict[str, Any]:
    # args: the arguments Claude passed in, matching the schema above
    # A custom tool must return this exact shape: {"content": [{"type": "text", "text": ...}]}
    price = get_stock_price(args["ticker"])
    return {"content": [{"type": "text", "text": f"{args['ticker']}: ${price['price']:.2f}"}]}


# create_sdk_mcp_server groups your tool(s) into a mini "server" object.
#   name    -> a label for this group of tools (shows up as "stocks" below)
#   version -> just a version string for the server
#   tools   -> the list of @tool-decorated functions this server offers
stock_server = create_sdk_mcp_server(name="stocks", version="1.0.0", tools=[get_stock_price_tool])

## Wire the tool into a session and ask about it


In [4]:
async def ask_stock_price() -> None:
    options = ClaudeAgentOptions(
        model="haiku",
        # mcp_servers: registers our custom tool server so Claude knows it exists.
        # The key "stocks" here is just a label matching what we named it above.
        mcp_servers={"stocks": stock_server},
        # allowed_tools: pre-approve this specific tool to run without asking.
        # Custom tool names always follow the pattern: mcp__<server_name>__<tool_name>
        allowed_tools=["mcp__stocks__get_stock_price"],
    )
    async with ClaudeSDKClient(options=options) as client:
        await client.query("What's the price of Tesla?")
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    # Watch Claude decide, on its own, to call our custom tool.
                    if isinstance(block, ToolUseBlock):
                        print(f"[tool call] {block.name}({block.input})")
            elif isinstance(message, ResultMessage):
                print(f"\nResult: {message.result}")


await ask_stock_price()

[tool call] ToolSearch({'query': 'select:mcp__stocks__get_stock_price', 'max_results': 1})
[tool call] mcp__stocks__get_stock_price({'ticker': 'TSLA'})

Result: The current price of Tesla (TSLA) is **$100.00**.


## Summary

- A custom tool is just a typed Python function wrapped by `@tool` and bundled with `create_sdk_mcp_server`.
- Wiring it in is two options fields: `mcp_servers` to register it, `allowed_tools` to permit it.
